# Linear regression 

Lets try to generate data in format 

y = 3x + 5 + noise

In [1]:
import torch
torch.manual_seed(2608)

In [15]:
X = torch.rand(100,1)*10
noise = torch.randn(100,1) * 1

# we do 100,1 meaning 100 rows and 1 column that means one feature. 
# doing just rand(100) will give us 100 rows and 1 column so 100 features and only one row

y = 3*X + 5 + noise

print(X.shape)
print(y.shape)

print(X[:5])
print(y[:5])

torch.Size([100, 1])
torch.Size([100, 1])
tensor([[2.2554],
        [9.6129],
        [9.9077],
        [4.4752],
        [2.1306]])
tensor([[ 9.9798],
        [34.0997],
        [33.8207],
        [17.6625],
        [11.0522]])


Lets initialize weights and bias to some random values and then try to learn the weights and bias using gradient descent.

In [26]:
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# we will try backprop thats why autograd. 

w,b

(tensor([-1.7179], requires_grad=True), tensor([0.0668], requires_grad=True))

In [27]:
y_pred = X * w + b

loss = torch.mean((y_pred - y) ** 2)

loss

tensor(1150.9254, grad_fn=<MeanBackward0>)

In [ ]:
# lets do backprop now 
# this will calculate dL/dw and dL/db

loss.backward()
# we can backward through loss only once. If we want to backward again we need to set retain_graph=True in loss.backward().
print(w.grad , b.grad)

tensor([-421.5766]) tensor([-62.3806])


now with gradients with use we can introduce learning rates

In [29]:
lr = 0.01

In [30]:
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad

    # we need to set the gradients to zero after updating weights
    w.grad.zero_()
    b.grad.zero_()

print(w,b)

tensor([2.4978], requires_grad=True) tensor([0.6907], requires_grad=True)


Let's put it altogether

In [36]:
X = torch.rand(100,2) * 10
noise = torch.randn(100,1) * 1

y = 3*X[:,0:1] + 5*X[:,1:2] + 5 + noise

w = torch.randn(2,1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.01

for epoch in range(1000):
    # forward
    y_pred = X @ w + b 

    # loss
    loss = torch.mean((y_pred - y) ** 2)

    # backward 
    loss.backward()

    # update params
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

        # we need to set the gradients to zero after updating weights
    w.grad.zero_()
    b.grad.zero_()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: w = {w.squeeze().tolist()}, b = {b.item()}, loss = {loss.item()}")



Epoch 0: w = [5.871228218078613, 5.0117926597595215], b = 0.6644259691238403, loss = 2848.787841796875
Epoch 100: w = [3.2862491607666016, 5.237192630767822], b = 1.6601029634475708, loss = 2.910771131515503
Epoch 200: w = [3.2063827514648438, 5.165913105010986], b = 2.5580365657806396, loss = 2.0841104984283447
Epoch 300: w = [3.146158218383789, 5.1121625900268555], b = 3.2351460456848145, loss = 1.6140494346618652
Epoch 400: w = [3.1007442474365234, 5.07163143157959], b = 3.7457358837127686, loss = 1.3467599153518677
Epoch 500: w = [3.0664994716644287, 5.041067600250244], b = 4.130756855010986, loss = 1.1947723627090454
Epoch 600: w = [3.040675640106201, 5.0180206298828125], b = 4.421090126037598, loss = 1.1083474159240723
Epoch 700: w = [3.021202802658081, 5.000641345977783], b = 4.640024185180664, loss = 1.0592055320739746
Epoch 800: w = [3.006518840789795, 4.98753547668457], b = 4.805118083953857, loss = 1.0312602519989014
Epoch 900: w = [2.9954464435577393, 4.977653503417969], b 

In [40]:
x_dash = torch.tensor([[4.0, 2.0]])
y_dash1 = x_dash @ w + b

y_dash1

with torch.no_grad():
    y_dash2 = x_dash @ w + b

y_dash2 , y_dash1

(tensor([[26.9119]]), tensor([[26.9119]], grad_fn=<AddBackward0>))

Let's try to make a class for this 

In [42]:
class LinearRegression:

    def __init__(self, n_features, lr=0.01):
        self.w = torch.randn(n_features, 1, requires_grad=True)
        self.b = torch.randn(1, requires_grad=True)
        self.lr = lr

    def forward(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        with torch.no_grad():
            return self.forward(X)

    def compute_loss(self, y_pred, y):
        return torch.mean((y_pred - y) ** 2)

    def backward(self, loss):
        loss.backward()

    def update_params(self):
        with torch.no_grad():
            self.w -= self.lr * self.w.grad
            self.b -= self.lr * self.b.grad

            # we need to set the gradients to zero after updating weights
            self.w.grad.zero_()
            self.b.grad.zero_()

    def fit(self, X, y, epochs=1000):
        for epoch in range(epochs):
            # forwards 
            y_pred = self.forward(X)

            # loss 
            loss = self.compute_loss(y_pred, y)

            # backward
            self.backward(loss)

            # update params
            self.update_params()

            # print loss every 100 epochs
            if epoch % 100 == 0:
                print(f"Epoch {epoch}: w = {self.w.squeeze().tolist()}, b = {self.b.item()}, loss = {loss.item()}")

    

In [43]:
model = LinearRegression(n_features=2, lr=0.01)

model.fit(X, y, epochs=1000)

model.predict(torch.tensor([[4.0, 2.0]]))

Epoch 0: w = [5.224245071411133, 5.574422836303711], b = 1.761075735092163, loss = 3045.2021484375
Epoch 100: w = [3.2128748893737793, 5.1717071533203125], b = 2.4850475788116455, loss = 2.1426522731781006
Epoch 200: w = [3.1510536670684814, 5.116532325744629], b = 3.1801064014434814, loss = 1.6473366022109985
Epoch 300: w = [3.104435920715332, 5.074925899505615], b = 3.704231023788452, loss = 1.3656895160675049
Epoch 400: w = [3.0692825317382812, 5.043552398681641], b = 4.099459171295166, loss = 1.2055351734161377
Epoch 500: w = [3.0427749156951904, 5.019893646240234], b = 4.397490978240967, loss = 1.1144675016403198
Epoch 600: w = [3.0227856636047363, 5.002053737640381], b = 4.622227668762207, loss = 1.0626848936080933
Epoch 700: w = [3.0077123641967773, 4.988601207733154], b = 4.791698932647705, loss = 1.0332391262054443
Epoch 800: w = [2.9963462352752686, 4.978456497192383], b = 4.919491291046143, loss = 1.016495943069458
Epoch 900: w = [2.9877750873565674, 4.970807075500488], b = 

tensor([[26.9435]])